In [1]:
from lumina import LUMINA
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

/home/huy/miniconda3/envs/myenv/lib/python3.9/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/huy/miniconda3/envs/myenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-2-7b-chat-hf')
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-chat-hf')

detector = LUMINA(model, tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.60s/it]


In [3]:
def read_json(file_path):
    data = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))

    return data
dolly_data = read_json("datasets/dolly/merged.jsonl")

correct_passages = dolly_data[0]["source_info"]["passages"].replace("\n", " ")
correct_passages = correct_passages.replace("  ", " ")

random_passages = dolly_data[10]["source_info"]["passages"].replace("\n", " ")
random_passages = random_passages.replace("  ", " ")

prompt = dolly_data[0]["prompt"].replace("\n", " ")
correct_instruction = prompt.replace("  ", " ")
random_instruction = correct_instruction.replace(correct_passages, random_passages)
print(random_instruction)
print(correct_instruction)

Briefly answer the following question: What are the names of the songs from The Story of June which don't have Chinese names? Bear in mind that your response should be strictly based on the following content:The earthquake occurred on a Tuesday morning and leveled almost all the buildings in Lone Pine and nearby settlements. Of the estimated 250–300 inhabitants of Lone Pine, 27 are known to have perished and 52 of the 59 houses were destroyed. One report states that the main buildings were thrown down in almost every town in Inyo County. About 130 kilometers (81 mi) south of Lone Pine, at Indian Wells, adobe houses sustained cracks. Property loss has been estimated at $250,000 (equivalent to $5,654,861 in 2021). As in many earthquakes, adobe, stone and masonry structures fared worse than wooden ones which prompted the closing of nearby Camp Independence which was an adobe structure destroyed in the quake. In case the passages do not contain the necessary information to answer the quest

In [4]:
prompt_w_context = correct_instruction
prompt_w_random_context = random_instruction
response = dolly_data[0]["response"]

# Returns (hallucination_score, mmd, ipr)
hallucination_score, mmd, ipr = detector.predict(
    prompt_w_context, 
    prompt_w_random_context, 
    response
)

TypeError: forward() got an unexpected keyword argument 'disturb_head_ids'